# TCL Phase 2 Kaggle Runs

Trains and evaluates the Phase 2 arms for one training dataset per session, as fixed in `docs/phase2/TCL-Phase2-Protocol.md`:

1. Arm C with lambda 0.1, 0.5, 1.0 (seed 0), then the lambda with the best validation AUROC for seeds 1 and 2.
2. Arm A (seed 0 also regenerates answers as the base model), B, and D, each with seeds 0, 1, 2.

Run two sessions in parallel: `TRAIN_DATASET = "triviaqa"` and `TRAIN_DATASET = "squad"`. Each writes a zip to `/kaggle/working/`.

Settings: **GPU T4 x2**, **Internet on**.

In [ ]:
REPO = "https://github.com/awabmoha/truth-calibration-layer.git"
COMMIT = "4170b3f325bcdf90acda8fac16209ab9e87cbd0c"
TRAIN_DATASET = "triviaqa"   # "triviaqa" or "squad"
LIMIT = None                 # e.g. 20 for a quick check
print(TRAIN_DATASET, LIMIT)

In [ ]:
import os, subprocess, json, shutil
if not os.path.exists("truth-calibration-layer"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("truth-calibration-layer")
subprocess.run(["git", "checkout", COMMIT], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
os.chdir("tcl_experiments")
subprocess.run(["pip", "install", "-q", "peft==0.21.2"], check=True)
# Kaggle ships torchao 0.10, which recent peft rejects; this project does not use torchao.
subprocess.run(["pip", "uninstall", "-y", "-q", "torchao"], check=True)
subprocess.run(["python", "scripts/check_runtime.py"], check=True)

## Base-model answers

Phase 2 trains and tests every arm on the base model's Phase 1 answers, which are committed in `runs/phase1_kaggle/qwen/`.

In [ ]:
for ds in ["triviaqa", "squad"]:
    path = f"runs/phase1_kaggle/qwen/{ds}/records.jsonl"
    with open(path) as f:
        print(ds, sum(1 for _ in f), "records")

## Runs

Every run is skipped if its `metrics.json` already exists, so the cell can be re-run after an interruption within the same session.

In [ ]:
OUT = f"/kaggle/working/phase2/{TRAIN_DATASET}"

def run(arm, seed, lam=None, regenerate=False):
    name = f"C-lam{lam}" if arm == "C" else arm
    out_dir = f"{OUT}/{name}/seed{seed}"
    if os.path.exists(f"{out_dir}/metrics.json"):
        print("skip", out_dir)
        return json.load(open(f"{out_dir}/metrics.json"))
    cmd = ["python", "scripts/train_trust_head.py", "--arm", arm, "--train-dataset", TRAIN_DATASET,
           "--seed", str(seed), "--out-dir", out_dir]
    if lam is not None:
        cmd += ["--lam", str(lam)]
    if regenerate:
        cmd += ["--regenerate"]
    if LIMIT:
        cmd += ["--limit", str(LIMIT), "--wikitext-limit", "5"]
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    return json.load(open(f"{out_dir}/metrics.json"))

# 1. Lambda selection for arm C (seed 0), by validation AUROC.
val_auc = {lam: run("C", 0, lam)["validation_auroc"] for lam in (0.1, 0.5, 1.0)}
best_lam = max(val_auc, key=lambda k: val_auc[k] if val_auc[k] is not None else -1)
print("validation AUROC by lambda:", val_auc, "-> chosen", best_lam)
json.dump({"validation_auroc": val_auc, "chosen": best_lam}, open(f"{OUT}/lambda_selection.json", "w"), indent=2)

# 2. Chosen lambda, seeds 1 and 2.
for seed in (1, 2):
    run("C", seed, best_lam)

# 3. Arms A, B, D with three seeds. A seed 0 regenerates answers as the base model.
for seed in (0, 1, 2):
    run("A", seed, regenerate=(seed == 0))
for arm in ("B", "D"):
    for seed in (0, 1, 2):
        run(arm, seed)

## Package output

In [ ]:
zip_path = shutil.make_archive(f"/kaggle/working/phase2_{TRAIN_DATASET}", "zip", "/kaggle/working/phase2")
print(zip_path, os.path.getsize(zip_path) // 1_000_000, "MB")